# Notebook of Reinforcement Learning for Bandit Problems
This notebook was developed to explain a bandit problem, which is formally equivalent to a one-state Markov Decision Process (MDP). It is defined by a tuple $(\mathcal{A}, p)$, where $\mathcal{A}$ is a finite set of actions, and $p$ represents the probability for the reward given the current action.

**Notes for this revision**

- `GaussianBandit` is now a plain educational class (not a `gymnasium.Env` subclass).
- `epsilon_greedy_policy` uses random tie-breaking so that equal Q-values are not biased to action 0.
- All stochastic routines accept a `seed` argument, and we provide `run_experiment` for multi-seed averaging.
- Policy-based contents have been moved to the **Appendix** at the bottom.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from ipywidgets import interact, fixed
%matplotlib inline


# Example of the bandit problem
There are five slot machines, and therefore, an agent has five discrete actions. The agent receives a scalar reward according to the choice of actions. The goal is to find an action that maximizes the expected reward.

In [ ]:
class GaussianBandit:
    """Simple multi-armed Gaussian bandit for educational use.

    This is intentionally NOT a gymnasium.Env subclass. For a bandit (one-state
    MDP) we only need a `step(action) -> reward` interface, and a concrete
    numpy-based random generator with a `seed` argument.
    """

    def __init__(self, seed=None):
        self.mu    = np.array([1.0, 5.0, 3.0, 2.0, 3.5])
        self.sigma = np.array([0.8, 0.9, 1.5, 1.5, 1.2])
        self.n_actions = len(self.mu)
        self.reset(seed=seed)

    # --- core API ---
    def reset(self, seed=None):
        self.rng = np.random.default_rng(seed)
        return None   # no observation (single state)

    def step(self, action):
        """Return a scalar reward drawn from N(mu[a], sigma[a])."""
        return float(self.rng.normal(self.mu[action], self.sigma[action]))

    # --- utilities ---
    @property
    def optimal_action(self):
        return int(np.argmax(self.mu))

    def plot(self):
        fig = plt.figure(figsize=(8, 5))
        axarr = fig.subplots(1, 1)
        x = np.arange(start=-3.0, stop=8.0, step=0.05)
        for index, mean in enumerate(self.mu):
            std = self.sigma[index]
            norm_pdf = stats.norm.pdf(x=x, loc=mean, scale=std)
            axarr.plot(x, norm_pdf, label='r%1.0f' % index)
        axarr.legend()
        axarr.set_xlabel('r')
        axarr.set_ylabel('pdf')
        axarr.grid()


## Visualization of the reward functions

In [ ]:
env = GaussianBandit(seed=0)
env.plot()
print('optimal action =', env.optimal_action)


# Action selection
## $\varepsilon$-greedy policy
$\varepsilon$-greedy is a method to balance exploration and exploitation in the value-based reinforcement learning. When the current estimate of the action-value function is $Q(a)$, the probability to select the action $a$ is given by
\begin{equation}
  \pi (a) =
  \begin{cases}
    \epsilon/\lvert A \rvert + (1 - \epsilon)/|\mathrm{argmax}\,Q| & \mathrm{if} \; \;
       a \in \mathrm{arg}\max_a Q(a), \\
    \epsilon/\lvert A \rvert & \mathrm{otherwise},
  \end{cases}
\end{equation}
where $\varepsilon \in [0, 1]$ is a hyperparameter and $\lvert A \rvert$ is the number of available actions.

**Teaching note.** A naive `argmax` always returns the first index when there are ties. This means that when all $Q(a)=0$ at the start, action 0 is structurally favored. We therefore use `argmax_random_tie` below so that ties are broken uniformly at random.

In [ ]:
def argmax_random_tie(x, rng=None):
    """Return an argmax of x, breaking ties uniformly at random."""
    if rng is None:
        rng = np.random.default_rng()
    x = np.asarray(x)
    max_val = np.max(x)
    best = np.flatnonzero(x == max_val)
    return int(rng.choice(best))


def epsilon_greedy_policy(Q, epsilon=0.1):
    """Epsilon-greedy policy (returns a probability vector over actions).

    With random tie-breaking among argmax actions.
    """
    n_actions = Q.shape[0]
    policy = epsilon / n_actions * np.ones(n_actions)
    max_val = np.max(Q)
    best = np.flatnonzero(Q == max_val)
    policy[best] += (1 - epsilon) / len(best)
    return policy


def plot_epsilon_greedy_policy(Q, epsilon=0.1):
    policy = epsilon_greedy_policy(Q, epsilon)
    action = np.arange(Q.shape[0])

    fig = plt.figure(figsize=(12, 5))
    axarr = fig.subplots(1, 2)
    axarr[0].bar(action, Q)
    axarr[0].set_xlabel('action a')
    axarr[0].set_ylabel('action value Q(a)')

    axarr[1].bar(action, policy, label=r'$\epsilon$ = %3.1f' % epsilon)
    axarr[1].set_xlabel('action a')
    axarr[1].set_ylabel(r'policy $\pi$ (a)')
    plt.legend()
    plt.show()


In [ ]:
# epsilon = 0.1 #@param {type:"slider", min:0, max:1, step:0.05}
Q = np.array([2.5, -2.5, 2.9, 1.2, 0.5])
interact(plot_epsilon_greedy_policy, Q=fixed(Q), epsilon=(0, 1, 0.05));


## Boltzmann distribution
The Boltzmann distribution is an alternative method for action selection. The probability distribution is defined by
\begin{equation}
  \pi (a) = \frac{ \exp (\beta Q(a)) }{\sum_{a'} \exp (\beta Q(a')) },
\end{equation}
where $\beta$ is a non-negative hyperparameter, which is often called the inverse temperature.

In [ ]:
def Boltzmann_policy(Q, beta=1.0):
    Qmax = np.max(Q)
    expQ = np.exp(beta * (Q - Qmax))
    policy = expQ / np.sum(expQ)
    return policy


def plot_Boltzmann_policy(Q, beta=1.0):
    policy = Boltzmann_policy(Q, beta)
    action = np.arange(Q.shape[0])

    fig = plt.figure(figsize=(12, 5))
    axarr = fig.subplots(1, 2)
    axarr[0].bar(action, Q)
    axarr[0].set_xlabel('action a')
    axarr[0].set_ylabel('action value Q(a)')

    axarr[1].bar(action, policy, label=r'$\beta$ = %3.1f' % beta)
    axarr[1].set_xlabel('action a')
    axarr[1].set_ylabel(r'policy $\pi$ (a)')
    plt.legend()
    plt.show()


In [ ]:
# beta = 1 #@param {type:"slider", min:0, max:5, step:0.2}
Q = np.array([2.5, -2.5, 2.9, 1.2, 0.5])
interact(plot_Boltzmann_policy, Q=fixed(Q), beta=(0, 8, 0.2));


# Value-based method

An agent learns an expected reward. When the agent selects an action $a$ and receives a scalar reward $r$, an estimate of the expected reward is updated by
\begin{equation*}
  Q(a) = Q(a) + \alpha (r - Q(a)),
\end{equation*}
where $\alpha$ is a learning rate. Below we provide a single-run routine that records $Q$-estimates and action counts at every step.

In [ ]:
def value_based_method(method,
                        beta=1.0,
                        epsilon=0.1,
                        number_of_steps=1000,
                        initial_Q=0.0,
                        seed=None,
                        show_plot=True):
    """Single-run bandit experiment.

    Parameters
    ----------
    method : {"epsilon", "Boltzmann"}
    beta : float
        Inverse temperature for Boltzmann policy.
    epsilon : float
        Exploration rate for epsilon-greedy.
    number_of_steps : int
    initial_Q : float or ndarray
        Initial value of the action-value function. Set to a large positive
        value (e.g. 10) to enable *optimistic initialization*.
    seed : int or None
        Random seed that controls both the environment and action sampling.
    show_plot : bool
        If True, draw the per-step Q and N curves.

    Returns
    -------
    dict with keys:
        Q : (n_actions, T+1) array
        N : (n_actions, T+1) array
        rewards : (T,) array
        actions : (T,) int array
        optimal_action : int
    """
    env = GaussianBandit(seed=seed)
    rng = np.random.default_rng(None if seed is None else seed + 10_000)

    n_actions = env.n_actions
    Q = np.zeros((n_actions, number_of_steps + 1))
    N = np.zeros((n_actions, number_of_steps + 1))
    Q[:, 0] = initial_Q
    rewards = np.zeros(number_of_steps)
    actions = np.zeros(number_of_steps, dtype=int)

    for t in range(number_of_steps):
        if method == 'epsilon':
            policy = epsilon_greedy_policy(Q[:, t], epsilon=epsilon)
        elif method == 'Boltzmann':
            policy = Boltzmann_policy(Q[:, t], beta=beta)
        else:
            raise ValueError(f'unknown method: {method}')
        action = int(rng.choice(n_actions, p=policy))
        reward = env.step(action)

        N[:, t+1] = N[:, t]
        N[action, t+1] += 1
        Q[:, t+1] = Q[:, t]
        alpha = 1.0 / N[action, t+1]
        Q[action, t+1] = Q[action, t] + alpha * (reward - Q[action, t])

        rewards[t] = reward
        actions[t] = action

    if show_plot:
        fig = plt.figure(figsize=(12, 5))
        axarr = fig.subplots(1, 2)
        for i in range(n_actions):
            axarr[0].plot(Q[i, :], label='$a%i$' % i)
        axarr[0].legend()
        axarr[0].set_xlabel('steps')
        axarr[0].set_ylabel('action value Q(a)')
        axarr[0].grid()
        for i in range(n_actions):
            axarr[1].plot(N[i, :], label='$N%i$' % i)
        axarr[1].legend()
        axarr[1].set_xlabel('steps')
        axarr[1].set_ylabel('N(a)')
        axarr[1].grid()
        plt.show()
        print('final Q:', Q[:, -1])
        print('average reward: %f' % rewards.mean())

    return dict(Q=Q, N=N, rewards=rewards, actions=actions,
                optimal_action=env.optimal_action)


In [ ]:
epsilon = 0.1 #@param {type:"slider", min:0, max:1, step:0.05}
_ = value_based_method('epsilon', epsilon=epsilon, seed=0)


In [ ]:
beta = 1 #@param {type:"slider", min:0, max:5, step:0.2}
_ = value_based_method('Boltzmann', beta=beta, seed=0)


## Multi-run averaging and comparison

A single run is noisy: the first few pulls dominate the learning trajectory. To evaluate exploration strategies fairly we average over many independent seeds and look at two standard curves:

- **Average reward per step** (how good actions are on average)
- **Optimal-action rate** (how often the best arm is chosen)

We also compare three configurations:

1. $\varepsilon$-greedy with $\varepsilon=0.1$ and $Q_0=0$ (exploration via policy)
2. greedy ($\varepsilon=0$) with $Q_0=10$ — **optimistic initialization** (exploration via init)
3. $\varepsilon=0.01$ baseline

In [ ]:
def run_experiment(method='epsilon',
                   n_runs=200,
                   number_of_steps=1000,
                   base_seed=0,
                   **kwargs):
    """Run `n_runs` independent bandit experiments and aggregate metrics.

    Returns
    -------
    avg_reward : (T,) array   -- per-step mean reward, averaged over runs
    opt_rate   : (T,) array   -- per-step optimal-action selection rate
    Q_last_run : (n_actions, T+1) array  -- Q trajectory of the last run
    """
    avg_reward_sum = None
    opt_action_sum = None
    last = None
    for k in range(n_runs):
        result = value_based_method(method,
                                    number_of_steps=number_of_steps,
                                    seed=base_seed + k,
                                    show_plot=False,
                                    **kwargs)
        if avg_reward_sum is None:
            avg_reward_sum = np.zeros(number_of_steps)
            opt_action_sum = np.zeros(number_of_steps)
        avg_reward_sum += result['rewards']
        opt_action_sum += (result['actions'] == result['optimal_action']).astype(float)
        last = result
    avg_reward = avg_reward_sum / n_runs
    opt_rate   = opt_action_sum / n_runs
    return avg_reward, opt_rate, last['Q']


def optimal_action_rate(actions, optimal_action):
    """Cumulative optimal-action selection rate."""
    hit = (np.asarray(actions) == optimal_action).astype(float)
    return np.cumsum(hit) / np.arange(1, len(hit) + 1)


In [ ]:
# Compare three strategies, averaged over n_runs seeds
n_runs = 200
T = 1000

configs = [
    dict(label=r'$\varepsilon$=0.1, $Q_0$=0',  method='epsilon', epsilon=0.1,  initial_Q=0.0),
    dict(label=r'$\varepsilon$=0.01, $Q_0$=0', method='epsilon', epsilon=0.01, initial_Q=0.0),
    dict(label=r'greedy, $Q_0$=10 (optimistic)', method='epsilon', epsilon=0.0,  initial_Q=10.0),
]

results = []
for cfg in configs:
    label = cfg.pop('label')
    avg_r, opt_r, Q_last = run_experiment(n_runs=n_runs, number_of_steps=T, **cfg)
    results.append((label, avg_r, opt_r, Q_last))

fig, axarr = plt.subplots(1, 2, figsize=(13, 5))
for label, avg_r, opt_r, _ in results:
    axarr[0].plot(avg_r, label=label)
    axarr[1].plot(opt_r, label=label)
axarr[0].set_xlabel('steps'); axarr[0].set_ylabel('average reward'); axarr[0].grid(); axarr[0].legend()
axarr[1].set_xlabel('steps'); axarr[1].set_ylabel('optimal action rate'); axarr[1].grid(); axarr[1].legend()
plt.tight_layout(); plt.show()

# And the per-arm Q trajectory of the last run (epsilon=0.1 run)
_, _, Q_last = results[0][1], results[0][2], results[0][3]
fig = plt.figure(figsize=(8, 5))
for i in range(Q_last.shape[0]):
    plt.plot(Q_last[i, :], label='$a%i$' % i)
plt.xlabel('steps'); plt.ylabel('Q(a)  (last run of $\varepsilon$=0.1)')
plt.grid(); plt.legend(); plt.show()


---
# Appendix（発展）

ここから下は**第1回の本編では扱いません**。興味のある人向けの発展内容です。

- 勾配降下法の復習
- 方策勾配型（policy-based）バンディット
- ベースライン導入による分散削減

本編では「価値ベースの更新」＋「ε-greedy / 楽観的初期値」までを目標にしています。

## Gradient descent

Consider the minimization problem: $\min_x J(x)$, where $J(x)$ is differentiable. The local optimum is obtained by applying gradient descent:
\begin{equation}
x = x - \alpha \nabla_{x} J(x),
\end{equation}
where $\nabla_{x} J(x)$ is a partial derivative of $J(x)$ with respect to $x$, and $\alpha$ is a non-negative stepsize hyperparameter.

In [ ]:
def eval_objective(x):
    Jx = 10 * np.power(x, 2)
    dJx = 20 * x
    return Jx, dJx


def optimize_by_gd(x_init, n_iterations, alpha):
    x = np.zeros(n_iterations + 1)
    x[0] = x_init
    y = np.zeros(n_iterations + 1)
    dy = np.zeros(n_iterations + 1)
    for i in range(n_iterations):
        y[i], dy[i] = eval_objective(x[i])
        x[i + 1] = x[i] - alpha * dy[i]
    y[-1], dy[-1] = eval_objective(x[-1])
    return x, y, dy


def plot_result(x, y, dy):
    xx = np.arange(-100, 100)
    Jx, _ = eval_objective(xx)

    fig = plt.figure(figsize=(12, 5))
    axarr = fig.subplots(1, 2)
    axarr[0].plot(xx, Jx)
    axarr[0].plot(x, y, 'ro-')
    axarr[0].set_xlabel('x')
    axarr[0].set_ylabel('J(x)')
    axarr[1].plot(dy, 'ro-')
    axarr[1].set_xlabel('iteration')
    axarr[1].set_ylabel('dJ(x)/dx')


### Example of gradient descent
The learning rate $\alpha$ affects convergence, and you can see that the algorithm never converges if $\alpha = 0.1$.

In [ ]:
alpha = 0.01 #@param {type:"slider", min:0.01, max:0.1, step:0.005}
x, y, dy = optimize_by_gd(x_init=-75, n_iterations=50, alpha=alpha)
plot_result(x, y, dy)


## Policy-based method
The Boltzmann distribution is chosen because the $\varepsilon$-greedy is not differentiable due to the max operator. Below we introduce a baseline (running average of rewards) to reduce variance, with the variable names kept separate for clarity.

In [ ]:
def policy_based_method(method='reduction',
                        alpha=0.05,
                        number_of_steps=1000,
                        seed=None,
                        show_plot=True):
    env = GaussianBandit(seed=seed)
    rng = np.random.default_rng(None if seed is None else seed + 10_000)

    n_actions = env.n_actions
    theta = np.zeros((n_actions, number_of_steps + 1))
    N = np.zeros((n_actions, number_of_steps + 1))

    running_reward_sum = 0.0   # cumulative sum of rewards up to step t
    avg_reward = 0.0            # running mean (= running_reward_sum / t)
    baseline = 0.0              # baseline used at the current update step

    for t in range(number_of_steps):
        policy = Boltzmann_policy(theta[:, t], beta=1.0)
        action = int(rng.choice(n_actions, p=policy))
        reward = env.step(action)

        N[:, t+1] = N[:, t]
        N[action, t+1] += 1

        gradient = np.identity(n_actions)[action] - policy
        if method == 'reduction':
            baseline = avg_reward        # use the previous running mean as baseline
            theta[:, t+1] = theta[:, t] + alpha * (reward - baseline) * gradient
        elif method == 'no reduction':
            theta[:, t+1] = theta[:, t] + alpha * reward * gradient
        else:
            raise ValueError(f'unknown method: {method}')

        running_reward_sum += reward
        avg_reward = running_reward_sum / (t + 1)

    if show_plot:
        fig = plt.figure(figsize=(12, 5))
        axarr = fig.subplots(1, 2)
        for i in range(n_actions):
            axarr[0].plot(theta[i, :], label='$a%i$' % i)
        axarr[0].legend(); axarr[0].set_xlabel('steps')
        axarr[0].set_ylabel('policy parameter theta'); axarr[0].grid()
        for i in range(n_actions):
            axarr[1].plot(N[i, :], label='$N%i$' % i)
        axarr[1].legend(); axarr[1].set_xlabel('steps')
        axarr[1].set_ylabel('N(a)'); axarr[1].grid()
        plt.show()
        print('average reward: %f' % avg_reward)

    return dict(theta=theta, N=N, avg_reward=avg_reward)


In [ ]:
alpha = 0.03 #@param {type:"slider", min:0.01, max:0.1, step:0.005}
_ = policy_based_method('no reduction', alpha=alpha, seed=0)


In [ ]:
alpha = 0.03 #@param {type:"slider", min:0.01, max:0.1, step:0.005}
_ = policy_based_method('reduction', alpha=alpha, seed=0)
